# Mini-tutorial: Pipelines and ColumnTransformers

In the end-to-end ML project we are about to use two scikit-learn tools that can look intimidating at first:

- `Pipeline`
- `ColumnTransformer`

The ideas are simpler than the syntax.

## Big picture

A **Pipeline** means:

> do step 1 → then step 2 → then step 3

A **ColumnTransformer** means:

> send different columns through different preprocessing steps

We'll use a tiny slice of the mystery rat biomechanics dataset so we can see the inputs and outputs directly.

## 0. Setup

Put `mystery_dataset_1.csv` in the same folder as this notebook.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

df = pd.read_csv("mystery_dataset_1.csv")

df.head()

# 1. Start with a tiny table

We'll use just a few columns so the transformations remain visible.

**Numerical measurements**
- `Belt_Speed`
- `Step_Length`
- `Mean_Ankle`

**Categorical labels**
- `Speed_Group`
- `Timepoint_Name`

Our eventual target will be `Step_Height`, but first we'll focus only on preprocessing.

In [ ]:
demo_cols = [
    "Belt_Speed",
    "Step_Length",
    "Mean_Ankle",
    "Speed_Group",
    "Timepoint_Name",
    "Step_Height",
]

demo = df[demo_cols].copy()

demo.head(8)

## Check for missing values

`Mean_Ankle` contains some missing values in this dataset.

That gives us something real to preprocess rather than inventing an example.

In [ ]:
demo.isna().sum()

# 2. What is a Pipeline?

Suppose we want to preprocess one numerical variable, `Mean_Ankle`.

We want to do two operations:

1. replace missing values with the median
2. standardize the result

Without a pipeline, we could do those steps manually.

Let's use only the first few rows so we can actually inspect the numbers.

In [ ]:
x = demo[["Mean_Ankle"]].head(10)

x

## Step 1: impute missing values

In [ ]:
imputer = SimpleImputer(strategy="median")

x_imputed = imputer.fit_transform(x)

pd.DataFrame(
    x_imputed,
    columns=["Mean_Ankle_imputed"]
)

`fit_transform()` did two things:

- **fit**: learned the median from the data
- **transform**: used that learned median to replace missing values

Next, scale the imputed values.

In [ ]:
scaler = StandardScaler()

x_scaled = scaler.fit_transform(x_imputed)

pd.DataFrame(
    x_scaled,
    columns=["Mean_Ankle_scaled"]
)

# 3. Put those sequential steps into one Pipeline

Instead of manually doing:

```text
impute
   ↓
scale
```

we can describe that sequence once.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_pipeline

Now the same two operations happen with one call:

In [ ]:
x_pipeline_output = numeric_pipeline.fit_transform(x)

pd.DataFrame(
    x_pipeline_output,
    columns=["Mean_Ankle_after_pipeline"]
)

### A Pipeline is just ordered plumbing

```text
raw Mean_Ankle
      ↓
SimpleImputer
      ↓
StandardScaler
      ↓
processed Mean_Ankle
```

Each step receives the output of the step before it.

# 4. Why do we need a ColumnTransformer?

Real ML tables contain different kinds of columns.

For example:

```text
Belt_Speed       → numeric
Step_Length      → numeric
Mean_Ankle       → numeric

Speed_Group      → categorical
Timepoint_Name   → categorical
```

We do **not** want to process all of those in the same way.

For numerical variables:

```text
impute → scale
```

For categorical variables:

```text
impute → one-hot encode
```

A `ColumnTransformer` lets us send each group of columns down its own branch.

In [ ]:
numeric_features = [
    "Belt_Speed",
    "Step_Length",
    "Mean_Ankle",
]

categorical_features = [
    "Speed_Group",
    "Timepoint_Name",
]

X_small = demo[
    numeric_features + categorical_features
].head(12)

X_small

# 5. Build the two preprocessing branches

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_pipeline

Now the categorical branch.

`OneHotEncoder` converts labels such as:

```text
speed16
speed20
speed24
```

into columns of 0s and 1s that a model can use.

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )),
])

categorical_pipeline

# 6. Combine the branches with ColumnTransformer

Read this as:

> apply `numeric_pipeline` to the numerical columns  
> apply `categorical_pipeline` to the categorical columns

In [ ]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features,
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])

preprocessor

The structure is now:

```text
                    ┌→ numeric columns → impute → scale ─────┐
raw DataFrame ──────┤                                        ├→ processed table
                    └→ categorical columns → impute → one-hot┘
```

Unlike a `Pipeline`, these branches happen **in parallel** on different columns.

# 7. Look at the actual output

In [ ]:
X_processed = preprocessor.fit_transform(X_small)

print("Input shape: ", X_small.shape)
print("Output shape:", X_processed.shape)

Why did the number of columns increase?

Because one categorical column such as `Speed_Group` turns into several one-hot columns.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

feature_names

In [ ]:
processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=X_small.index,
)

processed_df

Notice:

- numerical values have been standardized
- categorical labels have become 0/1 indicator columns
- the model no longer receives strings such as `"speed24"`

# 8. `fit()` versus `transform()`

### `fit()`

Learn something from the data:

- median used for imputation
- mean and standard deviation used for scaling
- categories used for one-hot encoding

### `transform()`

Use those already-learned rules on data.

In [ ]:
# Learn preprocessing rules from these rows
preprocessor.fit(X_small)

# Apply the SAME rules to new rows
X_new = demo[
    numeric_features + categorical_features
].iloc[20:25]

X_new_processed = preprocessor.transform(X_new)

pd.DataFrame(
    X_new_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_new.index,
)

This is why preprocessing should usually be **fit on the training data only**.

If we let the test set help determine medians, means, standard deviations, or categories, then information from the test set has leaked into model preparation.

# 9. Put preprocessing and the ML model into one Pipeline

A full ML workflow can itself be a pipeline:

```text
raw DataFrame
      ↓
ColumnTransformer
      ↓
prepared numerical matrix
      ↓
LinearRegression
      ↓
Step Height prediction
```

In [ ]:
model = Pipeline([
    ("preprocess", preprocessor),
    ("regression", LinearRegression()),
])

model

For this tiny demonstration we'll fit the model to a small subset.

This is **not** our proper train/test analysis yet. The goal is just to see that the full pipeline accepts the original mixed-type DataFrame directly.

In [ ]:
features = numeric_features + categorical_features

tiny = demo.dropna(subset=["Step_Height"]).head(500)

X_tiny = tiny[features]
y_tiny = tiny["Step_Height"]

model.fit(X_tiny, y_tiny)

predictions = model.predict(X_tiny.head(5))

pd.DataFrame({
    "observed_Step_Height": y_tiny.head(5).to_numpy(),
    "predicted_Step_Height": predictions,
})

# 10. Why pipelines are useful

Instead of remembering:

```text
impute these columns
scale these columns
encode those columns
keep track of the transformed matrix
then fit the model
```

we can give scikit-learn one object:

```python
model.fit(X_train, y_train)

model.predict(X_test)
```

and the same preprocessing steps are applied automatically and consistently.

That becomes especially important during cross-validation, because each training fold learns its **own** preprocessing parameters rather than accidentally learning from validation data.

# 11. Tiny summary

## Pipeline

Sequential operations:

```text
step A → step B → step C
```

## ColumnTransformer

Different operations for different columns:

```text
numeric columns     → numeric pipeline
categorical columns → categorical pipeline
```

## Together

```text
                       ┌→ numeric → impute → scale ──────┐
raw DataFrame ─────────┤                                  ├→ regression
                       └→ categorical → one-hot encode ──┘
```

# 12. One question before moving on

In this dataset, `Belt_Speed` and `Speed_Group` both describe treadmill speed.

Should we necessarily give **both** to the model?

That leads directly into the next section:

## A deliberately redundant representation of speed